# Select items for an annotation study

Use a CPU workspace. Pick the data you want to code, check the selection, and export a study draft with a working title. The draft contains the source table and exact item IDs. It carries no corpus text or media files.

The planned **Annotation Studies** Hub form will import this draft for questionnaire design, preview, and publication. This notebook currently exports the selection file. Keep outputs that display corpus content in your private workspace.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

TABLES = Path(os.environ.get("SEMINAR_OUTPUT", "/corpus/derived")) / "tables"
SOURCES = {
    "texts": ("Text ID", "Text Type"),
    "comments": ("Comment ID", None),
    "images": ("Image ID", "Image Type"),
}

## Choose a source and filters

Choose one source per study. Examples include text type `Caption` or `Transcription`, image type `Cover`, `Gallery`, or `Frame`, and collection strategy `FYP`, `search`, `hashtag`, or `profile`. The collection label and strategy must match the same collection entry. Leave a filter empty to consider all values.

Start with broad filters and inspect the results. For precise selection, paste the IDs you want into `only_ids` in the desired order, or list items to remove in `exclude_ids`. Keep IDs quoted as text.

In [ ]:
source = "texts"  # "texts", "comments", or "images"
working_title = ""

collection_label = ""
collection_strategy = ""
item_type = ""
text_contains = ""  # Literal text match for texts and comments
only_ids = []  # For example, ["123456789_caption", "987654321_caption"]
exclude_ids = []

id_column, type_column = SOURCES[source]
items = pd.read_csv(TABLES / f"{source}.csv", dtype=str, keep_default_na=False)
if items[id_column].str.strip().eq("").any() or items[id_column].duplicated().any():
    raise ValueError(f"{source}.csv must contain one row per non-empty {id_column}.")

selected = items
if collection_label or collection_strategy:
    memberships = pd.read_csv(TABLES / "collections.csv", dtype=str, keep_default_na=False)
    if collection_label:
        memberships = memberships[memberships["label"] == collection_label]
    if collection_strategy:
        memberships = memberships[memberships["strategy"] == collection_strategy]
    selected = selected[selected["Post ID"].isin(memberships["Post ID"])]
if item_type:
    if type_column is None:
        raise ValueError("Item type is available for texts and images.")
    selected = selected[selected[type_column] == item_type]
if text_contains:
    if "Text" not in selected:
        raise ValueError("Text search is available for texts and comments.")
    selected = selected[selected["Text"].str.contains(text_contains, case=False, regex=False)]
if only_ids:
    if len(set(only_ids)) != len(only_ids):
        raise ValueError("List each selected ID only once.")
    missing = set(only_ids) - set(selected[id_column])
    if missing:
        raise ValueError(f"Requested IDs missing from the filtered rows: {sorted(missing)}")
    selected = selected.set_index(id_column, drop=False).loc[only_ids]
selected = selected[~selected[id_column].isin(exclude_ids)]

draft_title = working_title.strip()
draft_source = source
draft_ids = selected[id_column].tolist()
preview_columns = [id_column, "Post ID", type_column, "Strategy", "Text"]
preview_columns = [column for column in preview_columns if column in selected]
print(f"Selected {len(draft_ids)} of {len(items)} {source} rows. Showing up to 25.")
display(selected.loc[:, preview_columns].head(25))

## Export the reviewed selection

Set `export_draft` to `True` after checking the selection and title. The file contains the selected IDs in order, even when the preview shows fewer rows. To export another selection, choose a new filename. Save this file alongside your study notes for import into the Hub form once available.

In [ ]:
export_draft = False
draft_path = Path.home() / "annotation-drafts" / "study-draft.json"

if export_draft:
    if not draft_title or not draft_ids:
        raise ValueError("Enter a working title and run the selection cell with at least one item.")
    draft_path.parent.mkdir(mode=0o700, parents=True, exist_ok=True)
    draft = {"version": 1, "title": draft_title, "source": draft_source, "item_ids": draft_ids}
    with draft_path.open("x", encoding="utf-8") as output:
        json.dump(draft, output, ensure_ascii=False, indent=2)
        output.write("\n")
    print(f"Saved {len(draft_ids)} item IDs to {draft_path}.")